# Classifying Commuter vs. Recreational Citi Bike Trips
### What Trip Features Predict Ride Purpose in New York City?

**Programming Tools for Urban Analytics** | Dr Qunshan Zhao  
**By:** Fabian Moreno

---

### Setup

In [8]:
#!pip install shap
#!pip install dotenv
#!pip install holidays

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.2 MB/s  0:00:00 eta 0:00:01


In [1]:
#  ----- Standard library 
import os
import sys
from pathlib import Path


# ----- Adding project root to the path
sys.path.insert(0, str(Path('..').resolve()))


# ----- Libraries for data manipualtion
import numpy as np
import pandas as pd
import duckdb

# ----- Libraries for APIs
import requests


# ----- Spatial libraries
import geopandas as gpd
import folium
from shapely.geometry import Point


# ----- Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')


# ----- libraries for machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import shap


# ----- Utility libraries
from dotenv import load_dotenv
from tqdm import tqdm
import holidays


# ----- Project modules
from src.utils import DATA_RAW, DATA_PRO, DB_PATH, log
load_dotenv()


# ----- DuckDB connection
con = duckdb.connect(str(DB_PATH))

print(f"DuckDB connected: {DB_PATH}")
print(f"DuckDB {duckdb.__version__}")

DuckDB connected: /Users/fabianmorenogarza/Documents/UrbanAnalytics/01_PTUA/citibike/data/citibike.duckdb
DuckDB 1.4.3


---
## I. Introduction

*(~400 words)*

**Research question:** *What trip-level features best predict whether a Citi Bike journey is made for commuting or recreational purposes, and what does the spatial and temporal distribution of classified trips reveal about urban mobility patterns in New York City?*

---
## II. Data Collection

*(~350 words)*

### 2.1 Download citibike data from S3

In [ ]:
from src.data_download import download_citibike_trips

# Download full year 2024 — skips January since it's already on disk
trip_files = download_citibike_trips(year=2024)

print({len(trip_files)})

In [2]:
# Verifying all files were downloaded correctly

files = sorted(DATA_RAW.glob("*-citibike-tripdata.parquet"))
total_rows = 0

for f in files:
    df = pd.read_parquet(f)
    total_rows += len(df)
    print(f"{f.name}: {len(df):,} rows")

202401-citibike-tripdata.parquet: 1,888,085 rows
202402-citibike-tripdata.parquet: 2,121,501 rows
202403-citibike-tripdata.parquet: 2,663,295 rows
202404-citibike-tripdata.parquet: 3,217,063 rows
202405-citibike-tripdata.parquet: 4,133,961 rows
202406-citibike-tripdata.parquet: 4,783,576 rows
202407-citibike-tripdata.parquet: 4,722,896 rows
202408-citibike-tripdata.parquet: 4,603,575 rows
202409-citibike-tripdata.parquet: 4,997,898 rows
202410-citibike-tripdata.parquet: 5,150,054 rows
202411-citibike-tripdata.parquet: 3,710,134 rows
202412-citibike-tripdata.parquet: 2,311,171 rows


In [3]:
# Quick look of the downloaded data
df_sample = pd.read_parquet(DATA_RAW / "202406-citibike-tripdata.parquet")
print(df_sample.shape)
print(df_sample.dtypes)
df_sample.head(3)

(4783576, 13)
ride_id                          str
rideable_type                    str
started_at            datetime64[us]
ended_at              datetime64[us]
start_station_name               str
start_station_id                 str
end_station_name                 str
end_station_id                   str
start_lat                    float64
start_lng                    float64
end_lat                      float64
end_lng                      float64
member_casual                    str
dtype: object


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,38F52721E98C6761,electric_bike,2024-06-19 19:02:23.487,2024-06-19 19:39:04.984,Pier 61 at Chelsea Piers,6233.04,Peck Slip & South St,5096.12,40.746872,-74.008210,40.707519,-74.001081,member
1,C8A3A961E2F7BB14,electric_bike,2024-06-25 07:52:20.777,2024-06-25 07:59:22.760,41 Ave & 77 St,6290.06,Tyler Ave & Maurice Ave,5881.01,40.745570,-73.888200,40.733940,-73.901280,member
2,90690D5942906B7C,classic_bike,2024-06-15 07:32:02.957,2024-06-15 07:33:32.205,Prospect Park SW & 10 Ave,3611.02,Windsor Pl & Howard Pl,3579.04,40.659945,-73.977504,40.659491,-73.980139,member


### 2.2 GBFS station metadata

In [4]:
from src.data_download import fetch_gbfs_stations

stations = fetch_gbfs_stations()
print(f"{len(stations)} stations")
print(stations.dtypes)
stations.head(3)

[16:32:16] [2.2] stations.parquet already on disk — loading
2360 stations
station_id        str
name              str
lat           float64
lon           float64
capacity        int64
region_id         str
dtype: object


,station_id,name,lat,lon,capacity,region_id
0,1905837242740508940,31 St & Broadway,40.762110,-73.925230,35,71
1,41495491-5d89-4e14-aab9-c3db04aad399,43 St & Skillman Ave,40.746927,-73.920825,19,71
2,2177014969129222184,Henry Hudson Pkwy E & W 231 St,40.883360,-73.915050,0,71


### 2.3 OpenWeatherMap API

1. Sign up at openweathermap.org/api
2. Add the key to your .env file when it arrives
3. Run fetch_weather_history()

### 2.4 NYC PLUTO land use data

In [6]:
import requests, io, pandas as pd


# print columns from NYC Pluto
r = requests.get("https://data.cityofnewyork.us/api/views/64uk-42ks/rows.csv?accessType=DOWNLOAD", timeout=180)
cols = pd.read_csv(io.StringIO(r.content.decode("utf-8")), nrows=0).columns.tolist()
print(cols)

['borough', 'Tax block', 'Tax lot', 'community board', 'census tract 2010', 'cb2010', 'schooldist', 'council district', 'postcode', 'firecomp', 'policeprct', 'healtharea', 'sanitboro', 'sanitsub', 'address', 'zonedist1', 'zonedist2', 'zonedist3', 'zonedist4', 'overlay1', 'overlay2', 'spdist1', 'spdist2', 'spdist3', 'ltdheight', 'splitzone', 'bldgclass', 'landuse', 'easements', 'ownertype', 'ownername', 'lotarea', 'bldgarea', 'comarea', 'resarea', 'officearea', 'retailarea', 'garagearea', 'strgearea', 'factryarea', 'otherarea', 'areasource', 'numbldgs', 'numfloors', 'unitsres', 'unitstotal', 'lotfront', 'lotdepth', 'bldgfront', 'bldgdepth', 'ext', 'proxcode', 'irrlotcode', 'lottype', 'bsmtcode', 'assessland', 'assesstot', 'exempttot', 'yearbuilt', 'yearalter1', 'yearalter2', 'histdist', 'landmark', 'builtfar', 'residfar', 'commfar', 'facilfar', 'borocode', 'BBL', 'condono', 'tract2010', 'xcoord', 'ycoord', 'latitude', 'longitude', 'zonemap', 'zmcode', 'sanborn', 'taxmap', 'edesignum', '

#### PLUTO columns to be kept

| Column | Why we need it |
|--------|----------------|
| `BBL` | Unique lot ID: identifies each tax lot |
| `landuse` | The key one: codes like `05` = commercial, `09` = park, `01` = residential. Used to tag stations as employment-anchored or recreational |
| `bldgclass` | Building type: extra detail on top of landuse |
| `zonedist1` | Zoning district: commercial / residential / mixed |
| `xcoord` / `ycoord` | NYC grid coordinates |
| `latitude` / `longitude` | Standard coordinates: used to spatially join stations to surrounding lots |
| `bct2020` | Census tract |
| `borough` | Manhattan, Brooklyn, Queens, Bronx, Staten Island |

In [ ]:
# Create a list for columns to be kept
keep_cols = [
    "BBL", "landuse", "bldgclass", "zonedist1",
    "xcoord", "ycoord", "latitude", "longitude",
    "bct2020", "borough",
]

df = df.dropna(subset=["latitude", "longitude"])

In [8]:
from importlib import reload
import src.data_download
reload(src.data_download)
from src.data_download import download_pluto

pluto = download_pluto(overwrite=True)
print(f"{len(pluto):,} lots")
print(pluto.dtypes)
pluto.head(3)

[16:50:40] [2.4] Downloading NYC PLUTO from NYC Open Data (~300 MB)...
[16:55:58] [2.4] Saved 857,161 PLUTO lots → pluto.parquet
857,161 lots
borough      str
zonedist1    str
bldgclass    str
landuse      str
BBL          str
xcoord       str
ycoord       str
latitude     str
longitude    str
bct2020      str
dtype: object


,borough,zonedist1,bldgclass,landuse,BBL,xcoord,ycoord,latitude,longitude,bct2020
0,QN,R3X,A5,1,4061730023,1047384,219559,40.7690896,-73.7720738,4112300
1,QN,R3X,A0,1,4061730024,1047363,219486,40.7688894,-73.7721503,4112300
2,QN,R2A,B3,1,4061690026,1046855,219159,40.7679955,-73.7739873,4112300


latitude, longitude, xcoord, and ycoord are still str instead of float64

In [9]:
# ----- 

# Fix coordinate dtypes
for col in ("latitude", "longitude", "xcoord", "ycoord"):
    pluto[col] = pd.to_numeric(pluto[col], errors="coerce")

pluto = pluto.dropna(subset=["latitude", "longitude"])

# Save fixed version
pluto.to_parquet(DATA_RAW / "pluto.parquet", index=False)

print(f"{len(pluto):,} lots")
print(pluto.dtypes)

857,161 lots
borough          str
zonedist1        str
bldgclass        str
landuse          str
BBL              str
xcoord         int64
ycoord         int64
latitude     float64
longitude    float64
bct2020          str
dtype: object


---
## III. DuckDB schema

In [10]:
from importlib import reload
import src.database
reload(src.database)
from src.database import create_schema, ingest_stations, ingest_trips, ingest_pluto, validate

create_schema(con)
ingest_stations(con)
ingest_trips(con)
ingest_pluto(con)
validate(con)

[16:59:42] [3.1] Creating DuckDB schema...
[16:59:42] [3.1] Schema created — 4 tables ready
[16:59:42] [3.2] Ingesting stations...
[16:59:42] [3.2] stations: 2,360 rows
[16:59:42] [3.2] Ingesting trips (this may take a minute)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[16:59:54] [3.2] trips: 44,303,209 rows
[16:59:54] [3.2] Ingesting PLUTO...
[16:59:55] [3.2] pluto: 857,161 rows
[16:59:55] [3.3] Validating...
[16:59:55]     stations: 2,360 rows
[16:59:55]     trips: 44,303,209 rows
[16:59:55]     pluto: 857,161 rows
[16:59:55]     Station ID match rate: 0.0% (0 / 44,303,209)
[16:59:55]     Trip date range: 2023-12-31 02:36:55.648000+00:00 -> 2024-12-31 23:57:46.390000+00:00
[16:59:55] [3.3] Validation complete


In [11]:
# 0.0% station ID match rate. That means trip start_station_id values aren't matching the station_id values in the stations table. Let's see why:

# Check what station IDs look like in each table
print("=== trips ===")
print(con.execute("SELECT start_station_id FROM trips LIMIT 5").df())

print("\n=== stations ===")
print(con.execute("SELECT station_id FROM stations LIMIT 5").df())

=== trips ===
  start_station_id
0          7407.13
1          7407.13
2          6526.01
3          6346.07
4          6364.07

=== stations ===
                             station_id
0                   1905837242740508940
1  41495491-5d89-4e14-aab9-c3db04aad399
2                   2177014969129222184
3                   2177016530523622712
4                   2177016940364559212


In [12]:
# IDs are in completely different formats. Let's try to see if we can match by name instead:

# Check if station names overlap
print("=== sample trip station names ===")
print(con.execute("SELECT DISTINCT start_station_name FROM trips LIMIT 5").df())

print("\n=== sample station names ===")
print(con.execute("SELECT name FROM stations LIMIT 5").df())

=== sample trip station names ===
        start_station_name
0      N 7 St & Driggs Ave
1     Lenox Ave & W 126 St
2  Bank St & Washington St
3           Great Jones St
4          E 65 St & 2 Ave

=== sample station names ===
                              name
0                 31 St & Broadway
1             43 St & Skillman Ave
2   Henry Hudson Pkwy E & W 231 St
3               Post Rd & W 251 St
4  David Sheridan Plaza & Broadway


In [13]:
# Both 'station' names and 'trip station' names have the same format.

# Verifying match rate with inner join
matched_names = con.execute("""
    SELECT COUNT(*) FROM trips t
    INNER JOIN stations s ON t.start_station_name = s.name
""").fetchone()[0]

total = con.execute("SELECT COUNT(*) FROM trips").fetchone()[0]
pct = matched_names / total * 100
print(f"Name match rate: {pct:.1f}% ({matched_names:,} / {total:,})")

Name match rate: 96.8% (42,888,023 / 44,303,209)


In [ ]:
# There is a very high match rate, the remaining names will be dropped for simplicity and scope purposes.

---
## III. Data Cleaning

In [14]:
# Loading all trips into a single dataframe using pandas
df = pd.concat([
    pd.read_parquet(f) for f in sorted(DATA_RAW.glob("*-citibike-tripdata.parquet"))
], ignore_index=True)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)


Shape: (44303209, 13)
Columns: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,62EF1AC5BE598131,classic_bike,2024-01-24 09:03:33.533,2024-01-24 09:06:53.535,E 102 St & 1 Ave,7407.13,E 103 St & Lexington Ave,7463.09,40.786995,-73.941648,40.790305,-73.947558,member
1,8464E543DAB27DBF,classic_bike,2024-01-30 08:21:29.510,2024-01-30 08:29:03.304,E 102 St & 1 Ave,7407.13,E 91 St & 2 Ave,7286.01,40.786995,-73.941648,40.781153,-73.949630,member
2,9C04FDC8549F5205,electric_bike,2024-01-22 21:18:25.199,2024-01-22 21:26:24.647,W 35 St & 8 Ave,6526.01,1 Ave & E 39 St,6303.01,40.752762,-73.992805,40.747140,-73.971130,member


#### 4.1 Summary statistics and missing values

In [15]:
# Overview of missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0')

print(summary)

                    missing_count  missing_pct
start_station_name          28675         0.06
start_station_id            28675         0.06
end_station_name           109040         0.25
end_station_id             117068         0.26
start_lat                   29835         0.07
start_lng                   29835         0.07
end_lat                    122304         0.28
end_lng                    122304         0.28


In [ ]:
# 'end_station' has more missing values than 'start_station'. This is 

Citi Bike introduced "ride anywhere" locking for e-bikes around 2023 — riders can lock an e-bike to any street sign or bike rack within the service area, not just at a dock. When that happens, the trip ends without a destination station, so end_station_name and end_lat/lng are null. Start station missings are much lower (0.06%) because you still need to unlock from somewhere.

In [16]:
# ----- verifying above statement with the data

# Check if missing end stations are concentrated in e-bike trips
missing_end = df[df['end_station_name'].isna()]
print("Rideable type breakdown for missing end station:")
print(missing_end['rideable_type'].value_counts(normalize=True).round(3))

print("\nRideable type breakdown for full dataset:")
print(df['rideable_type'].value_counts(normalize=True).round(3))


Rideable type breakdown for missing end station:
rideable_type
electric_bike    0.914
classic_bike     0.086
Name: proportion, dtype: float64

Rideable type breakdown for full dataset:
rideable_type
electric_bike    0.661
classic_bike     0.339
Name: proportion, dtype: float64


#### Missing Values

End station missings (0.25%) are disproportionately concentrated in electric bike trips:
- Electric bikes = 66% of all trips, but 91% of trips with missing end station
- This is consistent with Citi Bike's "ride anywhere" e-bike locking feature,
  which allows riders to lock e-bikes outside of docks
- These trips are dropped as they cannot be spatially matched to station land use,
  which is required for the proxy labeling heuristic in the next steps

#### 4.2 Cleaning

In [17]:
original_len = len(df)

# Calculate duration in minutes
df['duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# Drop trips with invalid duration (under 1 min or over 3 hours)
df = df[(df['duration_min'] >= 1) & (df['duration_min'] <= 180)]
print(f"After duration filter:     {len(df):>12,} rows (removed {original_len - len(df):,})")

# Drop rows missing station names (we join on name, not ID)
df = df.dropna(subset=['start_station_name', 'end_station_name'])
print(f"After station name filter: {len(df):>12,} rows (removed {original_len - len(df):,} total)")

# Drop rows missing coordinates
df = df.dropna(subset=['start_lat', 'start_lng', 'end_lat', 'end_lng'])
print(f"After coordinate filter:   {len(df):>12,} rows (removed {original_len - len(df):,} total)")

# Drop duplicate ride IDs
df = df.drop_duplicates(subset=['ride_id'])
print(f"After deduplication:       {len(df):>12,} rows (removed {original_len - len(df):,} total)")

print(f"Removed {original_len - len(df):,} rows total ({(original_len - len(df)) / original_len * 100:.2f}%)")
print(f"Clean dataset: {len(df):,} rows")

After duration filter:       44,229,301 rows (removed 73,908)
After station name filter:   44,136,037 rows (removed 167,172 total)
After coordinate filter:     44,123,031 rows (removed 180,178 total)
After deduplication:         44,123,031 rows (removed 180,178 total)
Removed 180,178 rows total (0.41%)
Clean dataset: 44,123,031 rows
